# The Koen & Kondlo (2009) mass function

One of the available mass functions in imf is an implementation of the "error-convolved power law" defined in Equations (3) and (5) of the [Koen & Kondlo (2009, KK09)](https://doi.org/10.1111/j.1365-2966.2009.14956.x) paper, "Fitting power-law distributions to data with measurement errors".

PDF, eqn. 3:
$$f_y(y) = \frac{1}{\sqrt{2\pi}\sigma} \frac{\gamma}{L^{-\gamma}-U^{-\gamma}} \int_L^U x^{-(\gamma+1)} \exp\left[-\frac{1}{2}\left(\frac{y-x}{\sigma}\right)^2\right]$$

CDF, eqn. 5:
$$F_y(y) = \Phi\left(\frac{y-U}{\sigma}\right)+\frac{1}{\sqrt{2\pi}\sigma}\frac{1}{L^{-\gamma}-U^{-\gamma}}\times\int_L^U \left(L^{-\gamma}-x^{-\gamma}\right) \exp\left[-\frac{1}{2}\left(\frac{y-x}{\sigma}\right)^2\right];$$

$$\Phi(v) = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^{v}e^{-t^2/2}dt$$

This function has to be integrated numerically.  It can therefore be prone to significant numerical error. (See [issue #12](https://github.com/keflavich/imf/issues/12) on the imf repository for an example.) 

As of [PR #38](https://github.com/keflavich/imf/pull/38), ``KoenConvolvedPowerLaw`` is more accurate. Instead of performing the KK09 integrals when called, ``KoenConvolvedPowerLaw`` constructs a lookup table of values at log-spaced points when instantiated and interpolates between these points.  

However, there are significant enough drawbacks to this method that we have elected to preserve the original infrastructure in a new MassFunction class, ``SpotKoenConvolvedPowerLaw`` (so named because it does all the integration "on the spot"). In this notebook, we demonstrate the differences between the two.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imf import KoenConvolvedPowerLaw, SpotKoenConvolvedPowerLaw
from time import perf_counter

In [ ]:
L = 0.03
U = 120
gamma = 0.9; alpha = gamma + 1
sigma = 0.4

In [ ]:
ntrials = 10

When instantiated, a `KoenConvolvedPowerLaw` is evaluated at a number
of points (default 200, but can be changed). `SpotKoenConvolvedPowerLaw` does not perform any integration on instantiation and is therefore faster to create.

.. note:: The power law components of the functions are defined 
through the exponent $\gamma=\alpha-1$ in the original paper. However, both implementations in `imf` accept $\alpha$ instead for consistency with the rest of the package.

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    spot_koc = SpotKoenConvolvedPowerLaw(L,U,alpha,sigma)
avg_spot_time = (perf_counter() - t0)/ntrials
print(f'Average time to create on-the-spot powerlaw: {avg_spot_time} s')

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    int_koc = KoenConvolvedPowerLaw(L,U,alpha,sigma)
avg_int_time = (perf_counter() - t0)/ntrials
print(f'Average time to create interpolated powerlaw: {avg_int_time} s')

In [ ]:
print(f'Interp/spot creation ratio: {avg_int_time/avg_spot_time}')

``SpotKoenConvolvedPowerLaw`` is slower to evaluate than ``KoenConvolvedPowerLaw``. The difference between the two is large, but in absolute time, evaluating both functions at a single point is still fairly fast.

In [ ]:
integral_form = False

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    spot_func = spot_koc(10,integral_form=integral_form)
avg_spot_time = (perf_counter() - t0)/ntrials
print(f'Average time to evaluate scalar on the spot: {avg_spot_time} s')

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    int_func = int_koc(10,integral_form=integral_form)
avg_int_time = (perf_counter() - t0)/ntrials
print(f'Average time to interpolate scalar: {avg_int_time} s')

In [ ]:
print(f'Interp/spot scalar evaluation ratio: {avg_int_time/avg_spot_time}')

However, if you need to evaluate the function at multiple points, the time difference can accumulate.

In [ ]:
m = np.geomspace(L,U,100)

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    spot_func = spot_koc(m,integral_form=integral_form)
avg_spot_time = (perf_counter() - t0)/ntrials
print(f'Average time to evaluate array on the spot: {avg_spot_time} s')

In [ ]:
t0 = perf_counter()
for n in range(ntrials):
    int_func = int_koc(m,integral_form=integral_form)
avg_int_time = (perf_counter() - t0)/ntrials
print(f'Average time to interpolate array: {avg_int_time} s')

In [ ]:
print(f'Interp/spot array evaluation ratio: {avg_int_time/avg_spot_time}')

## Accuracy

``KoenConvolvedPowerLaw`` relies on interpolation, which produces small inaccuracies. 

The default number of points in the lookup table is sufficient to capture the behavior of the mass function, but the values diverge from on-the-spot integration. However, they can be expected to remain within about .1% of the "true" values in most use cases.

In [ ]:
plt.plot(m,spot_func,label='on-the-spot')
plt.plot(m,int_func,label='interpolated')
plt.xscale('log'); plt.yscale('log')
plt.legend()
plt.xlabel(r'Mass ($M_\odot$)')
plt.ylabel('PDF');

In [ ]:
percent_diff = ((int_func-spot_func) / spot_func * 100)

plt.plot(m,percent_diff)
plt.xscale('log')
plt.title('Ratio')
plt.xlabel(r'Mass ($M_\odot$)')
plt.ylabel('Interpolated/On-the-spot difference (%)');

In summary, both implementations have strengths and weaknesses, and may be preferred for different scenarios. Here are the pros and cons for both:

|               | PRO         | CON |
| ------------- | -----------           | ----------- |
| Interpolation |       |     |
|               | Quick evaluation      | Slow instantiation       |
|               | Enables random sampling          | Diverges from on-the-spot        |
|               |         | Can cause incorrect optimal sampling (rarely)     |
| On-the-spot   |   |   |
|   | Quick instantiation | Slow evaluation |
|               | Good accuracy        | Cannot be randomly sampled |
